In [5]:
# #built in middlewares:
# 1. SummarizationMiddleware
# 2. Humaninloop
# 3. StructuredOutputMiddleware

%pip install langgraph



Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI

import os 
import dotenv
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
gemini = ChatGoogleGenerativeAI(api_key=api_key, model="gemini-3.5-flash")


agent = create_agent(
    model=gemini,
    checkpointer= InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=gemini,
            trigger=("messages", 10),
            keep=("messages", 5)
        ),
      
    ],
)

In [17]:
#run with thread if 

config = {"configurable": {"thread_id": "test-1"}} #this is used to identify the thread and all the
# messages in this thread will be summarized together. If you want to have different threads, you can use different thread_id.

In [18]:
questions ={
     'What is 2+2?',
     'what is 3X6?',
     "what is 10/2?",
     "what is 5-2?",
     "What is 2^3?",
     "What is the square root of 16?",
     
}



for q in questions:
    response = agent.invoke({'messages':[HumanMessage(content=q)]}, config=config)
    print(response)
    print(len(response['messages']))

{'messages': [HumanMessage(content='What is 2^3?', additional_kwargs={}, response_metadata={}, id='db017cb9-3d8a-479e-a980-c199b2462239'), AIMessage(content=[{'type': 'text', 'text': '2^3 is **8**. \n\n(2 × 2 × 2 = 8)', 'extras': {'signature': 'ErADCq0DAQw51seONmHhlvi/zQQTFs8QMV0nDTIvBAq1BoOyKnW2DriT+sw20eBL6leIT3uRHp8AX2L6w7Ec/ZbhtZfZhSVyPRY1zDT61zJnVHJVnaBUS/smPAPaqzc1forKBZX8rjHX/LdC354JHwet5EodH2KsSmjvFPbuRTTyNmpK96458kCT2P8VWJihyCCpd5JmktCyIEAtSTv4GGreuVfSNdHkfCegXitA9tyG7lO3M02K1xMZd+CxRbGyrZyTGYzjCXdv5OOSTKelc1pTkQRdWAX6JnTxtaJlbmx8rUNANd5SSoIQaLV5r+xw+MALYbn5P643OvqDztn6sJ7Gzc3vNUscYZlMqj4Nlt2MWpfj+xvMmA9sRBKV3f2xxO4f0yYirLAJmXZ4UQ5TIlMWZy6P7wfRrMNLDVrrnvOHkHfZvfzVzhjy+i/5Z9zz2QBcCvoDEh/YF9ijenCOyNstoc6Fq7DURHavVJw+RkmnDRMoFueNguV16BSpkXUVQBZhe1PDh3GfSeqqBcYYdU4iguLE2dixbOjK/G91qUDyJwxRC4VzpKyRtlSLse4EwvfC'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--

# Based on Token Size

In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
import os
import dotenv
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
gemini = ChatGoogleGenerativeAI(api_key=api_key, model="gemini-3.5-flash")



def search_hotels(city:str)-> str:
    """Search for hotels in a given city and return a list of hotel names."""
    # This is a mock implementation. In a real implementation, you would call an API or database to get the hotel information.
    hotels = {
        "New York": ["Hotel A", "Hotel B", "Hotel C"],
        "Los Angeles": ["Hotel D", "Hotel E", "Hotel F"],
        "Chicago": ["Hotel G", "Hotel H", "Hotel I"]
    }
    return hotels.get(city, [])


agent2 = create_agent(
    model=gemini,
    checkpointer= InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=gemini,
            tools=[search_hotels],
            trigger=("tokens", 550),
            keep=("tokens", 200)
        ),
      
    ],
)

config = {"configurable": {"thread_id": "test-2"}} #this is used to identify the thread and all the
def count_tokens(messages):
    total_tokens = sum(len(str(m.content)) for m in messages)
    return total_tokens// 4  # Assuming 1 token is approximately 4 characters


In [21]:
cities = ["New York", "Los Angeles", "Chicago", "Houston",
           "Phoenix"]

for c in cities:
    response = agent2.invoke({'messages':[HumanMessage(content=f"Find me hotels in {c}")]}, config=config)
    
    tokens = count_tokens(response['messages'])

    print(f"{c}: {tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 33.019781245s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

# HumanInLoop Middleware:

In [ ]:
from langhchain.agents import create_agent
from langchain.agents.middleware import HumanInLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
